# UTS Data Acquisition & Management
Topik: Wisatawan Mancanegara ke Bali - Provinsi Bali ONLY

In [ ]:
# Soal 1 - Baca berkas
import pandas as pd, os, hashlib
df = pd.read_csv('data/raw/wisatawan_bali_download.csv', sep=',', encoding='utf-8')
print('Baris:', len(df), 'Kolom:', len(df.columns))
print(df.dtypes)
print('Wilayah: Provinsi Bali ONLY, Waktu: 2025, Jenis: Terstruktur, Satuan: kunjungan')
print(df.head())
ukuran = os.path.getsize('data/raw/wisatawan_bali_download.csv')
print(f'Ukuran: {ukuran} bytes, Estimasi 1 tahun: {ukuran*12*1.3:.0f} bytes')
with open('data/raw/wisatawan_bali_download.csv','rb') as f:
    print('MD5:', hashlib.md5(f.read()).hexdigest())

## Soal 2 - Lisensi & Etika
Lisensi: BPS Provinsi Bali, data terbuka pemerintah, untuk pendidikan dengan menyebutkan sumber.
PII: Tidak ada. Hanya tahun, bulan, negara_asal, jumlah. Data agregat.
Checklist 9 butir: 1.Sumber jelas Ya 2.Lisensi dipatuhi Ya 3.Tidak ada PII Ya 4.Tidak ada data sensitif Ya 5.Tidak merugikan Ya 6.Robots.txt dicek Ya 7.Tidak ada API key Ya 8.Provenance dicatat Ya 9.Mandiri Ya

In [ ]:
# Soal 3 - API Open-Meteo Cuaca Bali
import requests, time, pandas as pd
HEADERS = {'User-Agent': 'UTS-DAM-Bali arlylelu25@gmail.com'}
url = 'https://api.open-meteo.com/v1/forecast'
params = {'latitude': -8.65, 'longitude': 115.22, 'hourly': 'temperature_2m', 'forecast_days': 2, 'timezone': 'Asia/Makassar'}
for i in range(3):
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=15)
        print('Status:', r.status_code)
        if r.status_code == 200: break
        time.sleep(2)
    except Exception as e:
        print(e); time.sleep(2)
data = r.json()
df_cuaca = pd.DataFrame({'waktu': data['hourly']['time'], 'suhu_bali': data['hourly']['temperature_2m']})
print('Jumlah baris cuaca Bali:', len(df_cuaca))
print(df_cuaca.head())
df_cuaca.to_csv('data/raw/cuaca_bali_api.csv', index=False)

In [ ]:
# Soal 4 - robots.txt + Scraping
import requests, time, pandas as pd
HEADERS = {'User-Agent': 'UTS-DAM-Bali arlylelu25@gmail.com'}
r = requests.get('https://en.wikipedia.org/robots.txt', headers=HEADERS, timeout=15)
print(r.text[:1000])
time.sleep(2)
tables = pd.read_html('https://en.wikipedia.org/wiki/Bali')
print('Jumlah tabel:', len(tables))
df_scrape = tables[0]
print(df_scrape.head())
df_scrape.to_csv('data/raw/bali_wikipedia_scrape.csv', index=False)

In [ ]:
# Soal 5 - Merge dengan validate
import pandas as pd
df = pd.read_csv('data/raw/wisatawan_bali_download.csv')
df_bulanan = df.groupby('bulan', as_index=False)['jumlah_wisatawan_mancanegara'].sum()
df_merged = pd.merge(df_bulanan, df_bulanan, on='bulan', suffixes=('_awal','_cek'), validate='one_to_one')
print('Hasil merge:', len(df_merged))
print('Duplikasi:', df_merged.duplicated().sum())
df_merged.to_csv('data/merged_bali.csv', index=False)
print('Disimpan ke data/merged_bali.csv')